In [8]:
import os
from fastmcp import Client

In [9]:
mcp_servers = {

    # https://github.com/modelcontextprotocol/servers/tree/main/src/filesystem
    "filesystem": {
      "command": "npx",  # npx (strumento di Node.js) scarica ed esegue automaticamente il codice del server MCP come processo locale,
                         # avviandolo anche se non è già installato sulla macchina.
                         # In questo modo il server viene lanciato “al volo” e rimane attivo per la durata della sessione.
      "transport": "stdio", # possiamo usare stdio perché il processo è attivo sulla nostra macchina
      "args": [
        "-y",
        "@modelcontextprotocol/server-filesystem",
        "/home/simone/Maya/MCP",
      ]
    },

    # https://github.com/TimLukaHorstmann/mcp-weather
    "weather": {
      "command": "npx",
       "transport": "stdio", 
      "args": ["-y", "@timlukahorstmann/mcp-weather"],
      "env": { "ACCUWEATHER_API_KEY": os.getenv("accuweather_api_key") }
    }
}

In [19]:
async with Client(mcp_servers) as client:
    tools = await client.list_tools()
    for tool in tools:
        print(f"{tool.name}: {tool.description} \n")

filesystem_read_file: Read the complete contents of a file from the file system. Handles various text encodings and provides detailed error messages if the file cannot be read. Use this tool when you need to examine the contents of a single file. Only works within allowed directories. 

filesystem_read_multiple_files: Read the contents of multiple files simultaneously. This is more efficient than reading files one by one when you need to analyze or compare multiple files. Each file's content is returned with its path as a reference. Failed reads for individual files won't stop the entire operation. Only works within allowed directories. 

filesystem_write_file: Create a new file or completely overwrite an existing file with new content. Use with caution as it will overwrite existing files without warning. Handles text content with proper encoding. Only works within allowed directories. 

filesystem_edit_file: Make line-based edits to a text file. Each edit replaces exact line sequences

In [16]:
async with Client(mcp_servers) as client:
    res = await client.call_tool("filesystem_read_file", {"path": "test_file.txt"})
    print(res.content[0].text)

Ciao io sono Simone e questo è un video sul Model Context Protocol


In [23]:
import os
from agents import Agent, Runner
from agents.mcp import MCPServerStdio

async def run_personal_assistant(prompt: str):
    async with MCPServerStdio(
        params={
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-filesystem", "/home/simone/Maya/MCP"],
        },
        name="filesystem",
    ) as filesystem, \
    MCPServerStdio(
        params={
            "command": "npx",
            "args": ["-y", "@timlukahorstmann/mcp-weather"],
            "env": {"ACCUWEATHER_API_KEY": os.getenv("accuweather_api_key")},
        },
        name="weather",
    ) as weather:

        agent = Agent(
            name="AssistentePersonale",
            instructions=(
                "Sei un assistente personale che aiuta l'utente nella gestione quotidiana. "
                "Usa i tool disponibili se necessario."
            ),
            mcp_servers=[filesystem, weather],
            model="gpt-4o",
        )

        result = await Runner.run(starting_agent=agent, input=prompt)
        return result.final_output


output = await run_personal_assistant("Che tempo ci sarà domani oggi a Roma?")
print(output)


Oggi a Roma il tempo sarà parzialmente nuvoloso durante il giorno e nuvoloso di notte, con temperature che varieranno tra 18°C e 27.7°C.
